# DocuRoute — train Layer 2 on Colab

Local training is a fallback only: the target laptop is an i3-1215U with 8 GB
of RAM and about 1 GB usually free, which measured ~86 min per fold. Five
folds is over seven hours there, and minutes on a T4.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

### How this run is organised

The five folds exist **only to measure performance**, so their weights are
thrown away — a fold keeps its metrics and its per-example predictions, about
18 KB instead of 254 MB. Those files go to Google Drive as each fold finishes,
so a disconnect costs one fold rather than the run, and re-running skips
whatever already completed.

Fusion trains from those saved prediction files, here or locally — it never
needs the fold models.

Only the **final model**, trained on every document at the end, is downloaded.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/DocuRoute/context_v2')
(DRIVE / 'folds').mkdir(parents=True, exist_ok=True)
(DRIVE / 'final').mkdir(parents=True, exist_ok=True)
print('results will be written to', DRIVE)

## 2. Clone the repository

Colab pulls from GitHub, so whatever you are training must be **pushed** first.

In [ ]:
REPO = 'https://github.com/Doculan/DocuRoute1.git'
BRANCH = 'main'

import os, shutil
if os.path.exists('/content/DocuRoute1'):
    shutil.rmtree('/content/DocuRoute1')
!git clone --depth 1 --branch {BRANCH} {REPO} /content/DocuRoute1
%cd /content/DocuRoute1/Backend
!git -C /content/DocuRoute1 log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

## 3. Build the splits

`all.jsonl` and `folds.json` are committed; the per-fold splits are rebuilt
here rather than uploaded.

In [ ]:
import pathlib

if pathlib.Path('ml/revision_pipeline/scripts/make_splits.py').exists():
    !python ml/revision_pipeline/scripts/make_splits.py --folds all
else:
    print('make_splits.py not present yet (Phase 7) - upload split files manually')

for name in ('train', 'val', 'test'):
    p = pathlib.Path(f'ml/datasets/context_v2/{name}.jsonl')
    print(f'{name:<6}', sum(1 for _ in p.open(encoding='utf-8')) if p.exists() else 'MISSING')

## 4. Cross-validation folds — metrics only

`--no-save-weights` keeps `metrics.json`, `predictions.jsonl` and
`thresholds.json` and discards the encoder. Each fold is copied to Drive as
soon as it finishes, and a fold already present in Drive is skipped.

In [ ]:
import json, pathlib, shutil, subprocess, time

FOLDS = list(range(5))
EPOCHS = 3
BATCH = 16
MAX_LENGTH = 384

for fold in FOLDS:
    drive_fold = DRIVE / 'folds' / f'fold_{fold}'
    if (drive_fold / 'metrics.json').exists():
        print(f'fold {fold}: already in Drive, skipping')
        continue

    data_dir = pathlib.Path(f'ml/datasets/context_v2/fold_{fold}')
    if not data_dir.exists():
        data_dir = pathlib.Path('ml/datasets/context_v2')
    out_dir = pathlib.Path(f'ml/saved_models/context_v2/fold_{fold}')

    print(f'\n=== fold {fold} ===')
    started = time.time()
    subprocess.run([
        'python', 'ml/revision_pipeline/scripts/train_layer2.py',
        '--data-dir', str(data_dir), '--out-dir', str(out_dir),
        '--epochs', str(EPOCHS), '--batch-size', str(BATCH),
        '--max-length', str(MAX_LENGTH), '--device', 'cuda',
        '--fold', str(fold), '--no-save-weights',
    ], check=True)

    drive_fold.mkdir(parents=True, exist_ok=True)
    for name in ('metrics.json', 'predictions.jsonl', 'thresholds.json', 'training_log.csv'):
        src = out_dir / name
        if src.exists():
            shutil.copy2(src, drive_fold / name)
    print(f'fold {fold} finished in {(time.time() - started) / 60:.1f} min -> {drive_fold}')

print('\nfolds present in Drive:',
      sorted(p.name for p in (DRIVE / 'folds').glob('fold_*')))

### Fold summary

In [ ]:
import json

for fold_dir in sorted((DRIVE / 'folds').glob('fold_*')):
    metrics = json.loads((fold_dir / 'metrics.json').read_text())
    last = metrics['epochs'][-1] if metrics.get('epochs') else {}
    print(f"{fold_dir.name}: best_epoch={metrics.get('best_epoch')} "
          f"verdict_macro_f1={last.get('val_verdict_macro_f1')} "
          f"issues_micro_f1={last.get('val_issues_micro_f1')}")

## 5. Fusion, from the saved predictions

Fusion reads the fold prediction files, so this cell runs equally well on a
laptop after downloading the `folds/` directory.

In [ ]:
import pathlib

if pathlib.Path('ml/revision_pipeline/scripts/train_fusion.py').exists():
    !python ml/revision_pipeline/scripts/train_fusion.py --predictions-dir "{DRIVE}/folds" --out-dir ml/saved_models/context_v2
else:
    print('train_fusion.py not present yet (Phase 3)')

## 6. The final model — trained on every document

This is the only model that is kept. It trains on all documents, holding out a
small slice purely so **early stopping** has something to watch. Per-label
thresholds are taken as the **median across the five folds** instead,
since 8% of the data cannot tune a rare label. Saved to Drive as one zip.

In [ ]:
import json, pathlib, random, shutil, subprocess

FINAL_DIR = pathlib.Path('ml/saved_models/context_v2/final')
FINAL_DATA = pathlib.Path('ml/datasets/context_v2/_final')
FINAL_DATA.mkdir(parents=True, exist_ok=True)

# Prefer all.jsonl; otherwise stitch the splits back together.
all_path = pathlib.Path('ml/datasets/context_v2/all.jsonl')
if all_path.exists():
    rows = [json.loads(l) for l in all_path.open(encoding='utf-8') if l.strip()]
else:
    rows = []
    for name in ('train.jsonl', 'val.jsonl', 'test.jsonl'):
        p = pathlib.Path('ml/datasets/context_v2') / name
        if p.exists():
            rows += [json.loads(l) for l in p.open(encoding='utf-8') if l.strip()]

random.Random(42).shuffle(rows)
cut = max(int(len(rows) * 0.08), 1)
(FINAL_DATA / 'val.jsonl').write_text('\n'.join(json.dumps(r) for r in rows[:cut]), encoding='utf-8')
(FINAL_DATA / 'train.jsonl').write_text('\n'.join(json.dumps(r) for r in rows[cut:]), encoding='utf-8')
print(f'final model: {len(rows) - cut} train / {cut} held out for early stopping')

subprocess.run([
    'python', 'ml/revision_pipeline/scripts/train_layer2.py',
    '--data-dir', str(FINAL_DATA), '--out-dir', str(FINAL_DIR),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH),
    '--max-length', str(MAX_LENGTH), '--device', 'cuda',
    # Thresholds come from the folds' median, not from the 8% slice:
    # that slice is far too small to tune a rare label on.
    '--thresholds-from', str(DRIVE / 'folds'),
], check=True)

for item in sorted(FINAL_DIR.iterdir()):
    print(' ', item.name)

## 7. Save the final model to Drive

Unzip locally into **`Backend/ml/saved_models/context_v2/`** so that the
folder contains `encoder/`, `tokenizer/`, `heads.pt`, `thresholds.json` and
`label_config.json`.

`saved_models/` is gitignored — the encoder alone is ~254 MB, past what GitHub
accepts without LFS. `ml/reports/` is small and **is** committed, so copy those
into the repo and commit them.

In [ ]:
import pathlib, shutil

archive = shutil.make_archive('/content/context_v2_final', 'zip', FINAL_DIR)
size_mb = pathlib.Path(archive).stat().st_size / 1e6
shutil.copy2(archive, DRIVE / 'final' / 'context_v2_final.zip')
print(f'{size_mb:.0f} MB -> {DRIVE / "final" / "context_v2_final.zip"}')

reports = pathlib.Path('ml/reports')
if reports.exists():
    shutil.make_archive('/content/reports', 'zip', reports)
    shutil.copy2('/content/reports.zip', DRIVE / 'final' / 'reports.zip')
    print('reports copied to Drive')

# Optional direct download as well as the Drive copy.
# from google.colab import files; files.download(archive)